## Task 1: Writing code to pass tests

Consider the following test cases:

In [ ]:
import pytest
from calculator import Calculator

# Constants

NUMBER_1 = 3.0
NUMBER_2 = 2.0


# Fixtures

@pytest.fixture
def calculator():
    return Calculator()


# Helper

def verify_answer(expected, answer, last_answer):
    assert expected == answer
    assert expected == last_answer


# Test Cases

def test_last_answer_init(calculator):
    assert calculator.last_answer == 0.0


def test_add(calculator):
    answer = calculator.add(NUMBER_1, NUMBER_2)
    verify_answer(5.0, answer, calculator.last_answer)


def test_subtract(calculator):
    answer = calculator.subtract(NUMBER_1, NUMBER_2)
    verify_answer(1.0, answer, calculator.last_answer)


def test_subtract_negative(calculator):
    answer = calculator.subtract(NUMBER_2, NUMBER_1)
    verify_answer(-1.0, answer, calculator.last_answer)


def test_multiply(calculator):
    answer = calculator.multiply(NUMBER_1, NUMBER_2)
    verify_answer(6.0, answer, calculator.last_answer)


def test_divide(calculator):
    answer = calculator.divide(NUMBER_1, NUMBER_2)
    verify_answer(1.5, answer, calculator.last_answer)


def test_divide_by_zero(calculator):
    with pytest.raises(ZeroDivisionError) as e:
        calculator.divide(NUMBER_1, 0)
    assert "division by zero" in str(e.value)


@pytest.mark.parametrize("a,b,expected", [
    (NUMBER_1, NUMBER_2, NUMBER_1),
    (NUMBER_2, NUMBER_1, NUMBER_1),
    (NUMBER_1, NUMBER_1, NUMBER_1),
])
def test_maximum(calculator, a, b, expected):
    answer = calculator.maximum(a, b)
    verify_answer(expected, answer, calculator.last_answer)


@pytest.mark.parametrize("a,b,expected", [
    (NUMBER_1, NUMBER_2, NUMBER_2),
    (NUMBER_2, NUMBER_1, NUMBER_2),
    (NUMBER_2, NUMBER_2, NUMBER_2),
])
def test_minimum(calculator, a, b, expected):
    answer = calculator.minimum(a, b)
    verify_answer(expected, answer, calculator.last_answer)

**How to proceed:**
- We need to follow the methodology.
1. Create a test module called `test_calculator.py` and place the test cases below in it
2. Run `pytest -v test_calculator.py` to see your tests failing. Note reasons given
3. Create an application module, `calculator.py` and in it create a class **Calculator**
4. Decide on the nature of your constructor. Then run pytest
5. Add your first method to enable your first test case to pass. Then run the test.

**Questions:**
- what does the fixture do?
- what does the helper do?

**What do the fixtures do?**

### Task 2: Wallet Scenario

A wallet application enables users to `add` or `spend` cash that is in the wallet. We want to create a wallet as a Python class with methods `spend_cash()` and `add_cash()`.

**Write tests first**

Create a test module called `test_wallet.py`. Place the following code snippet in it:

In [ ]:
import pytest

def test_default_initial_amount():
    wallet = Wallet()
    assert wallet.balance == 0

def test_setting_initial_amount():
    wallet = Wallet(100)
    assert wallet.balance == 100

def test_wallet_add_cash():
    wallet = Wallet(10)
    wallet.add_cash(90)
    assert wallet.balance == 100

def test_wallet_spend_cash():
    wallet = Wallet(20)
    wallet.spend_cash(10)
    assert wallet.balance == 10

def test_wallet_spend_cash_raises_exception_on_insufficient_amount():
    wallet = Wallet()
    with pytest.raises(InsufficientAmount):
        wallet.spend_cash(100)

Study your test code carefully. Run the test  - `pytest -v test_wallet.py`

Now turn your attention to the application code.

Create a module called `wallet.py`

These are pointers as to what to write in `wallet.py`

- a class called Wallet
- the constructor should have an argument called `initial_amount` that is initialised to 0
- provide instance variable `balance` that is set to `initial_amount` i.e. `self.balance = initial_amount`
- provide a method `add_cash` that takes an `amount` argument. This is the amount by which you change the balance, i.e. `self.balance = self.balance + amount`
- povide a `spend-cash` method that takes an argument named `amount`. This is the amount being taken from the wallet, hence reducing the balance. Check that if `balance<amount` raise the InsufficientAmount exception with an appropriate message e.g. raise InsufficientAmount('Not enough cash in wallet'). Otherwise, reduce balance by amount

-  you will need to create a class called `InsufficientAmount` that inherits `Exception`. In the body of the class, do nothing i.e. `pass`. Place InsufficientAmount class above the Wallet class.

Import your classes from `wallet.py` module.

**Now run your tests again**

- do `pytest -v test_wallet.py` 
- do `pytest -q test_wallet.py`



**Let's refactor the tests using fixtures**


We have some repetition  - we initialized the class in each test.

We sort this out with pytest fixtures. We want to set up some helper code that should run before any tests are executed.


We modify the `test_wallet.py` and add two fixtures as follows:

In [ ]:
import pytest
from wallet import Wallet, InsufficientAmount

@pytest.fixture
def empty_wallet():
    '''Returns a Wallet instance with a zero balance'''
    return Wallet()

@pytest.fixture
def wallet():
    '''Returns a Wallet instance with a balance of 20'''
    return Wallet(20)

def test_default_initial_amount(empty_wallet):
    assert empty_wallet.balance == 0

def test_setting_initial_amount(wallet):
    assert wallet.balance == 20

def test_wallet_add_cash(wallet):
    wallet.add_cash(80)
    assert wallet.balance == 100

def test_wallet_spend_cash(wallet):
    wallet.spend_cash(10)
    assert wallet.balance == 10

def test_wallet_spend_cash_raises_exception_on_insufficient_amount(empty_wallet):
    with pytest.raises(InsufficientAmount):
        empty_wallet.spend_cash(100)

Describe what the above code does?

Is it better than our first version?

How do fixtures work?


### Useful information - on fixtures

- Each test is provided with a newly-initialized Wallet instance, and not one that has been used in another test.
- It is good practice to add docstrings for your fixtures. 

Run the following command to see all fixtures:

### Parametrizing test functions
We have tested the individual methods in the Wallet class. 
But we ought to test various data permuations for these methods. We want to answer questions such as -  `if I have an initial balance of 30, and spend 20, then add 100, and later on, spend 50, how much should the balance be?`.

Writing such a test will reuquire a number of tedious steps. `pytest` solves it by parameterizing test functions with arguments and data.


In the refactored `test_wallet.py` module, add this test function

In [ ]:
@pytest.mark.parametrize("earned,spent,expected", [
    (30, 10, 20),
    (20, 2, 18),
])
def test_transactions(earned, spent, expected):
    my_wallet = Wallet()
    my_wallet.add_cash(earned)
    my_wallet.spend_cash(spent)
    assert my_wallet.balance == expected

### What have we just done?

- we are able to test different scenarios in one  function. 
- use `@pytest.mark.parametrize` decorator - we specify names of arguments that will be passed to the test function, and a list of  corresponding  values.

- The test function marked with the decorator will run once for each set of parameters.

**Example** -  the test will be run the first time with the earned parameter set to `30`, spent set to `10`, and expected set to `20`. Second time the test is run, the parameters will take the second set of arguments. 


**What scenario of tests have we captured?**



- wallet initially has 0,
- add 30 units of cash to the wallet,
- spend 10 units of cash, and
- should have 20 cash remaining after the two transactions.

### Can we combine fixtures with parameterization?

**Yes we can**

We should aim to make tests less repetitive. We may combine test `fixtures` and `parametrize` test functions. For example, let’s replace the wallet initialization code with a test fixture as before.

In [ ]:
@pytest.fixture
def my_wallet():
    '''Returns a Wallet instance with a zero balance'''
    return Wallet()

@pytest.mark.parametrize("earned,spent,expected", [
    (30, 10, 20),
    (20, 2, 18),
])
def test_transactions(my_wallet, earned, spent, expected):
    my_wallet.add_cash(earned)
    my_wallet.spend_cash(spent)
    assert my_wallet.balance == expected

### What have we just done?

- We created a new fixture  `my_wallet` that is the same as the `empty_wallet` fixture we previously used. It returns a wallet instance with a balance of 0. 
- To use both the fixture and the parametrized functions in the test, we include the fixture as the first argument and the parameters as the rest of the arguments.

- The transactions are  performed on the wallet instance provided by the fixture.

**Try it a bit more...**
-  with the wallet instance having non-empty balance and with other different combinations of the earned and spent amounts.



## Task 3: Testing and mocking a shopping cart

This exercise covers various pytest features including:
- Basic assertions and test structure
- Fixtures and dependency injection
- Parameterized testing
- Test organization
- Mocking external dependencies
- Testing exceptions
- Fixture scoping
- Marking tests

**You're to implement a shopping cart system with the following components:**
1. Product catalog
2. Shopping cart functionality
3. Checkout process with discounts
4. External payment gateway integration

**How to go about it:**

1.Start with tests (test classes and their test cases), run these tests
2. Implement the required classes and functions, then run the tests again
3. Write comprehensive refactored tests (if neecessary) 
4. Ensure all tests pass

In all cases, use `pytest`

Here is sample test code to get you started. The main work is writing code for one test at time and check that tests are passing.
Place the following test code in a module named `test_shoppingcart.py`. Your application code should be placed in a file called `shoppingcart.py`

In [ ]:
import pytest
from unittest.mock import MagicMock, patch

from shoppingcart import ShoppingCart, Product, ProductCatalog, PaymentGateway\
    , PercentageDiscount, CategoryDiscount, CheckoutService


@pytest.fixture
def product_catalog():
    catalog = ProductCatalog()
    # Add some test products
    catalog.add_product(Product(1, "Laptop", 1000, "Electronics"))
    catalog.add_product(Product(2, "Headphones", 100, "Electronics"))
    catalog.add_product(Product(3, "T-shirt", 20, "Clothing"))
    catalog.add_product(Product(4, "Jeans", 50, "Clothing"))
    catalog.add_product(Product(5, "Python Book", 35, "Books"))
    return catalog


@pytest.fixture
def cart(product_catalog):
    return ShoppingCart(product_catalog)


@pytest.fixture
def filled_cart(cart):
    cart.add_item(1)  # Laptop
    cart.add_item(2, 2)  # 2 Headphones
    cart.add_item(3, 3)  # 3 T-shirts
    return cart


@pytest.fixture
def mock_payment_gateway():
    gateway = MagicMock(spec=PaymentGateway)
    # Configure the mock to return success for valid card
    gateway.process_payment.return_value = {"success": True, "transaction_id": "t123456789"}
    return gateway


# ----- Test Product Catalog -----

class TestProductCatalog:
    def test_add_and_get_product(self, product_catalog):
        # Add a new product
        new_product = Product(6, "Coffee Mug", 10, "Home")
        product_catalog.add_product(new_product)

        # Retrieve and verify
        retrieved = product_catalog.get_product(6)
        assert retrieved.id == 6
        assert retrieved.name == "Coffee Mug"
        assert retrieved.price == 10
        assert retrieved.category == "Home"

    def test_get_nonexistent_product(self, product_catalog):
        with pytest.raises(KeyError) as excinfo:
            product_catalog.get_product(999)
        assert "Product with id 999 not found" in str(excinfo.value)

    def test_get_products_by_category(self, product_catalog):
        electronics = product_catalog.get_products_by_category("Electronics")
        assert len(electronics) == 2
        assert electronics[0].name in ["Laptop", "Headphones"]
        assert electronics[1].name in ["Laptop", "Headphones"]

        clothing = product_catalog.get_products_by_category("Clothing")
        assert len(clothing) == 2

        # Test category with no products
        empty = product_catalog.get_products_by_category("Food")
        assert len(empty) == 0

    def test_search_products(self, product_catalog):
        # Case insensitive search
        results = product_catalog.search_products("lap")
        assert len(results) == 1
        assert results[0].name == "Laptop"

        # Multiple results
        results = product_catalog.search_products("on")
        assert len(results) == 2  # Python Book and Headphones

        # No results
        results = product_catalog.search_products("xyz")
        assert len(results) == 0


# ----- Test Shopping Cart -----

class TestShoppingCart:
    def test_add_item(self, cart):
        cart.add_item(1)
        items = cart.get_items()
        assert len(items) == 1
        assert items[0]["product"].id == 1
        assert items[0]["quantity"] == 1

    def test_add_item_with_quantity(self, cart):
        cart.add_item(1, 3)
        items = cart.get_items()
        assert items[0]["quantity"] == 3

    def test_add_item_incrementing_quantity(self, cart):
        cart.add_item(1, 2)
        cart.add_item(1, 3)
        items = cart.get_items()
        assert items[0]["quantity"] == 5

    def test_add_invalid_item(self, cart):
        with pytest.raises(KeyError):
            cart.add_item(999)

    def test_add_invalid_quantity(self, cart):
        with pytest.raises(ValueError) as excinfo:
            cart.add_item(1, 0)
        assert "Quantity must be positive" in str(excinfo.value)

        with pytest.raises(ValueError):
            cart.add_item(1, -1)

    def test_remove_item(self, filled_cart):
        filled_cart.remove_item(1)
        items = filled_cart.get_items()
        assert len(items) == 2
        assert all(item["product"].id != 1 for item in items)

    def test_remove_item_with_quantity(self, filled_cart):
        # Initially 2 headphones
        filled_cart.remove_item(2, 1)
        items = {item["product"].id: item["quantity"] for item in filled_cart.get_items()}
        assert items[2] == 1

    def test_remove_item_not_in_cart(self, cart):
        with pytest.raises(KeyError):
            cart.remove_item(1)

    def test_update_quantity(self, filled_cart):
        filled_cart.update_quantity(1, 10)
        items = {item["product"].id: item["quantity"] for item in filled_cart.get_items()}
        assert items[1] == 10

    def test_update_quantity_to_zero_removes_item(self, filled_cart):
        filled_cart.update_quantity(1, 0)
        items = filled_cart.get_items()
        assert all(item["product"].id != 1 for item in items)

    def test_get_total(self, filled_cart):
        # 1 Laptop (£1000) + 2 Headphones (£100 each) + 3 T-shirts (£20 each) = £1260
        assert filled_cart.get_total() == 1260

    def test_clear_cart(self, filled_cart):
        filled_cart.clear()
        assert len(filled_cart.get_items()) == 0
        assert filled_cart.get_total() == 0


# ----- Test Discount Rules -----

class TestDiscountRules:
    def test_percentage_discount(self, filled_cart):
        # 10% off
        discount = PercentageDiscount(10)
        # Cart total is £1260, so discount should be £126
        assert discount.apply(filled_cart) == 126

    def test_percentage_discount_with_minimum(self, filled_cart):
        # 10% off orders over £2000
        discount = PercentageDiscount(10, 2000)
        # Cart total is $1260, so no discount
        assert discount.apply(filled_cart) == 0

        # Add more items to reach minimum
        filled_cart.add_item(1, 1)  # Another laptop (£1000)
        # Total is now £2260, so discount should be £226
        assert discount.apply(filled_cart) == 226

    def test_category_discount(self, filled_cart):
        # 20% off Electronics
        discount = CategoryDiscount("Electronics", 20)
        # Electronics in cart: 1 Laptop (£1000) + 2 Headphones (£200) = £1200 * 20% = £240
        assert discount.apply(filled_cart) == 240

        # 50% off Clothing
        discount = CategoryDiscount("Clothing", 50)
        # Clothing in cart: 3 T-shirts (£60) * 50% = £30
        assert discount.apply(filled_cart) == 30

    def test_invalid_percentage(self):
        with pytest.raises(ValueError):
            PercentageDiscount(110)

        with pytest.raises(ValueError):
            CategoryDiscount("Books", -10)


# ----- Test Checkout Service -----

class TestCheckoutService:
    @pytest.fixture
    def checkout_with_discounts(self, filled_cart, mock_payment_gateway):
        discounts = [
            PercentageDiscount(10),  # 10% off total
            CategoryDiscount("Electronics", 5)  # 5% off electronics
        ]
        return CheckoutService(filled_cart, discounts, mock_payment_gateway)

    def test_calculate_total_no_discounts(self, filled_cart):
        checkout = CheckoutService(filled_cart)
        assert checkout.calculate_total() == 1260

    def test_calculate_total_with_discounts(self, checkout_with_discounts):
        # Discounts:
        # - 10% off total: £1260 * 10% = £126
        # - 5% off electronics: (£1000 + £200) * 5% = £60
        # Total discount: £186
        # Final total: £1260 - £186 = £1074
        assert checkout_with_discounts.calculate_total() == 1074

    def test_successful_checkout(self, checkout_with_discounts):
        result = checkout_with_discounts.checkout({
            "card_number": "1111222233334444",
            "expiry": "12/25",
            "cvv": "123"
        })

        assert result["success"] is True
        assert result["total"] == 1074
        assert result["transaction_id"] == "t123456789"

        # Cart should be cleared after successful checkout
        assert len(checkout_with_discounts.cart.get_items()) == 0

In [ ]:
# shoppingcart.py - the solution

# ─────────────────────────────────────────────────────────────────────────────
# shoppingcart.py
# A  shopping cart implementation covering:
#   Product, ProductCatalog, ShoppingCart, discount rules, and CheckoutService
# ─────────────────────────────────────────────────────────────────────────────


# ── Product ───────────────────────────────────────────────────────────────────

class Product:
    """A single product in the catalogue."""

    def __init__(self, id: int, name: str, price: float, category: str):
        self.id = id
        self.name = name
        self.price = price
        self.category = category

    def __repr__(self):
        return f"Product({self.id}, {self.name!r}, £{self.price}, {self.category!r})"


# ── ProductCatalog ────────────────────────────────────────────────────────────

class ProductCatalog:
    """Stores all available products and provides look-up helpers."""

    def __init__(self):
        self._products: dict[int, Product] = {}   # keyed by product id for O(1) look-up

    def add_product(self, product: Product) -> None:
        """Add a product to the catalogue."""
        self._products[product.id] = product

    def get_product(self, product_id: int) -> Product:
        """Return a product by id, or raise KeyError if not found."""
        if product_id not in self._products:
            raise KeyError(f"Product with id {product_id} not found")
        return self._products[product_id]

    def get_products_by_category(self, category: str) -> list[Product]:
        """Return all products that belong to the given category."""
        return [p for p in self._products.values() if p.category == category]

    def search_products(self, query: str) -> list[Product]:
        """Case-insensitive search across product names."""
        query_lower = query.lower()
        return [p for p in self._products.values() if query_lower in p.name.lower()]


# ── ShoppingCart ──────────────────────────────────────────────────────────────

class ShoppingCart:
    """
    Holds items a customer intends to buy.
    Each item is stored as {"product": Product, "quantity": int}.
    """

    def __init__(self, catalog: ProductCatalog):
        self._catalog = catalog
        self._items: dict[int, dict] = {}   # keyed by product_id for easy look-up

    # ── mutation ──────────────────────────────────────────────────────────────

    def add_item(self, product_id: int, quantity: int = 1) -> None:
        """
        Add `quantity` of a product to the cart.
        Raises KeyError if the product doesn't exist in the catalogue.
        Raises ValueError if quantity is not a positive integer.
        """
        if quantity <= 0:
            raise ValueError("Quantity must be positive")

        product = self._catalog.get_product(product_id)   # raises KeyError if missing

        if product_id in self._items:
            self._items[product_id]["quantity"] += quantity   # increment existing line
        else:
            self._items[product_id] = {"product": product, "quantity": quantity}

    def remove_item(self, product_id: int, quantity: int = None) -> None:
        """
        Remove an item from the cart.
        If quantity is given, reduce by that amount (removing the line if it hits 0).
        If quantity is None, remove the entire line regardless of current quantity.
        Raises KeyError if the product is not in the cart.
        """
        if product_id not in self._items:
            raise KeyError(f"Product {product_id} is not in the cart")

        if quantity is None:
            del self._items[product_id]   # remove the whole line
        else:
            self._items[product_id]["quantity"] -= quantity
            if self._items[product_id]["quantity"] <= 0:
                del self._items[product_id]   # drop the line if quantity reaches zero

    def update_quantity(self, product_id: int, quantity: int) -> None:
        """
        Set the quantity of an item directly.
        Setting quantity to 0 removes the item from the cart entirely.
        """
        if product_id not in self._items:
            raise KeyError(f"Product {product_id} is not in the cart")

        if quantity == 0:
            del self._items[product_id]
        else:
            self._items[product_id]["quantity"] = quantity

    def clear(self) -> None:
        """Remove all items from the cart."""
        self._items.clear()

    # ── read-only helpers ─────────────────────────────────────────────────────

    def get_items(self) -> list[dict]:
        """Return all cart lines as a list of {"product": ..., "quantity": ...} dicts."""
        return list(self._items.values())

    def get_total(self) -> float:
        """Return the sum of (price × quantity) for every line in the cart."""
        return sum(
            item["product"].price * item["quantity"]
            for item in self._items.values()
        )


# ── Discount Rules ────────────────────────────────────────────────────────────

class PercentageDiscount:
    """
    Applies a flat percentage discount to the entire cart total.
    Optionally requires a minimum cart value before the discount kicks in.
    """

    def __init__(self, percentage: float, minimum_order: float = 0):
        if not (0 <= percentage <= 100):
            raise ValueError(f"Percentage must be between 0 and 100, got {percentage}")
        self.percentage = percentage
        self.minimum_order = minimum_order

    def apply(self, cart: ShoppingCart) -> float:
        """Return the discount amount (0 if the cart total is below minimum_order)."""
        total = cart.get_total()
        if total < self.minimum_order:
            return 0                                  # minimum not reached - no discount
        return round(total * self.percentage / 100, 2)


class CategoryDiscount:
    """
    Applies a percentage discount only to items that belong to a specific category.
    """

    def __init__(self, category: str, percentage: float):
        if not (0 <= percentage <= 100):
            raise ValueError(f"Percentage must be between 0 and 100, got {percentage}")
        self.category = category
        self.percentage = percentage

    def apply(self, cart: ShoppingCart) -> float:
        """Return the discount amount calculated only on matching category items."""
        category_total = sum(
            item["product"].price * item["quantity"]
            for item in cart.get_items()
            if item["product"].category == self.category
        )
        return round(category_total * self.percentage / 100, 2)


# ── PaymentGateway ────────────────────────────────────────────────────────────

class PaymentGateway:
    """
    Represents an external payment processor (e.g. Stripe, PayPal).
    In tests this is replaced with a MagicMock so no real money moves.
    """

    def process_payment(self, amount: float, payment_details: dict) -> dict:
        """
        Submit a payment request to the external gateway.
        Returns {"success": bool, "transaction_id": str}.
        In production this would make a real API call.
        """
        raise NotImplementedError("Implement with a real payment provider")


# ── CheckoutService ───────────────────────────────────────────────────────────

class CheckoutService:
    """
    Orchestrates the checkout flow:
      1. Calculate the discounted total
      2. Charge the customer via the payment gateway
      3. Clear the cart on success
    """

    def __init__(self, cart: ShoppingCart,
                 discounts: list = None,
                 payment_gateway: PaymentGateway = None):
        self.cart = cart
        self.discounts = discounts or []          # list of discount rule objects
        self.payment_gateway = payment_gateway    # can be None for total-only calculations

    def calculate_total(self) -> float:
        """
        Return the cart total after applying all discount rules.
        Each discount rule's apply() returns a discount amount; we subtract them all.
        """
        gross = self.cart.get_total()
        total_discount = sum(d.apply(self.cart) for d in self.discounts)
        return round(gross - total_discount, 2)

    def checkout(self, payment_details: dict) -> dict:
        """
        Charge the customer and, on success, clear the cart.
        Returns the gateway response enriched with the final total.
        Raises RuntimeError if no payment gateway has been configured.
        """
        if self.payment_gateway is None:
            raise RuntimeError("No payment gateway configured")

        total = self.calculate_total()

        # Delegate the actual charge to the gateway (mocked in tests)
        result = self.payment_gateway.process_payment(total, payment_details)

        if result.get("success"):
            self.cart.clear()   # only clear the cart after a confirmed successful payment

        # Merge our total into the gateway response so the caller gets one clean dict
        return {**result, "total": total}